In [ ]:
# IMAGE TO TEXT USING GOOGLE VISION API (BULK PROCESSING)

from google.cloud import vision
from pathlib import Path
import os
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = r"credentials.json"

def ocr_image(image_path: str):
    client = vision.ImageAnnotatorClient()

    with open(image_path, "rb") as f:
        content = f.read()

    image = vision.Image(content=content)

    # DOCUMENT_TEXT_DETECTION is better for scanned docs
    response = client.document_text_detection(image=image)

    if response.error.message:
        raise Exception(response.error.message)

    text = response.full_text_annotation.text
    return text


if __name__ == "__main__":
    input_folder = r"input_1"
    
    # Create an output folder alongside the input folder
    output_folder = f"{input_folder}_TXT"
    os.makedirs(output_folder, exist_ok=True)
    
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".pdf"}
    
    for filename in os.listdir(input_folder):
        file_path = os.path.join(input_folder, filename)
        
        if os.path.isfile(file_path):
            ext = os.path.splitext(filename)[1].lower()
            if ext in valid_exts:
                # Same filename, but with .txt extension
                base_name = os.path.splitext(filename)[0]
                txt_filename = f"{base_name}.txt"
                txt_path = os.path.join(output_folder, txt_filename)
                
                if os.path.exists(txt_path):
                    print(f"Skipping {filename}, already processed.")
                    continue
                
                print(f"Processing: {filename}")
                try:
                    text = ocr_image(file_path)
                    with open(txt_path, "w", encoding="utf-8") as f:
                        f.write(text)
                    print(f"Saved: {txt_filename}")
                except Exception as e:
                    print(f"Error processing {filename}: {e}")

In [ ]:
# MAIN PROCESSING CODE OCR + LLM
# PAN Number extracted via REGEX (reliable fixed format)
# Aadhaar details extracted via LLM (Mistral)
# Aadhaar NUMBER extracted via REGEX (primary) — no more LLM for digits

import os
import re
import json
import pandas as pd
from pathlib import Path
from difflib import SequenceMatcher
from datetime import datetime

import ollama

current_timestamp = datetime.now().strftime("%Y-%m-%d %H-%M-%S")

# =========================
# CONFIG
# =========================

INPUT_FILE = r"input_file.xlsx"
OUTPUT_FILE = f"final_output4{current_timestamp}.xlsx"
OUTPUT_FILE_path = OUTPUT_FILE

DMS_FOLDER = Path(r"input_1_TXT")


# =========================
# HELPERS
# =========================

def get_txt_filename(name):
    if not name or name == "N/A":
        return None
    name = str(name).strip()
    name_lower = name.lower()
    for ext in (".jpg", ".jpeg", ".png"):
        if name_lower.endswith(ext):
            name = name[:-len(ext)]
            break
    if not name.lower().endswith(".txt"):
        name += ".txt"
    return name


def normalize_name(name):
    if not name or name == "N/A":
        return "N/A"
    name = re.sub(r"[^A-Za-z\s]", "", str(name)).upper().strip()
    return " ".join(sorted(name.split()))


def normalize_dob(d):
    if not d or d == "N/A":
        return "N/A"
    d = str(d).replace(" 00:00:00", "").strip()
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y", "%Y/%m/%d"):
        try:
            return datetime.strptime(d, fmt).strftime("%d/%m/%Y")
        except:
            continue
    return d


def normalize_address(addr):
    if not addr or addr == "N/A":
        return "N/A"
    try:
        if isinstance(addr, dict):
            return " ".join(str(v) for v in addr.values())
        if isinstance(addr, str) and addr.startswith("{"):
            data = json.loads(addr.replace("'", '"'))
            return " ".join(str(v) for v in data.values())
    except:
        pass
    return str(addr)


def match_pct(a, b):
    if not a or not b or a == "N/A" or b == "N/A":
        return 0
    return round(SequenceMatcher(None, str(a), str(b)).ratio() * 100, 2)


# =========================
# READ TEXT FILE
# =========================

def read_text_file(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.read().strip()
    except:
        try:
            with open(path, "r", encoding="latin-1") as f:
                return f.read().strip()
        except Exception as e:
            print(f"  READ FAIL: {path} - {e}")
            return ""


# =========================
# REGEX: AADHAAR NUMBER (PRIMARY METHOD)
# =========================

def extract_aadhaar_regex(text):
    """
    Extracts Aadhaar number directly from OCR text using regex.
    This is the PRIMARY method — far more reliable than asking LLM for digits.

    Handles all real-world patterns seen in your data:

    UNMASKED (12 digits):
      "8931 5795 4230"          -> spaced groups of 4
      "893157954230"            -> compact no spaces
      "9807 1207 4580"          -> Subhashree old letter format

    MASKED (4 digits only):
      "XXXX XXXX 6941"          -> X-masked with last 4
      "6941" standalone line    -> Tinu Verma style
      "8627" standalone line    -> Mohit Gupta style

    Excludes:
      VID lines (16 digits)     -> stripped before matching
      Dates, phone numbers      -> length/pattern guards
      1947 UIDAI helpline       -> stripped before matching
    """

    if not text:
        return "N/A", "masked"

    # ── Pre-clean: strip VID lines, download/issue dates, helpline ──
    clean = text

    # Remove VID lines: "VID : 9185 4467 3464 6701" or "VID: 917527600330 2013"
    clean = re.sub(r'VID\s*[:\.]?\s*[\d\s]{15,25}', '', clean, flags=re.IGNORECASE)
    # Remove Virtual ID lines
    clean = re.sub(r'.*[Vv]irtual\s*[Ii][Dd].*', '', clean)
    # Remove Download/Issue Date lines
    clean = re.sub(r'(Download|Issue|Print)\s*Date\s*[:\.]?\s*[\d/\-]+', '', clean, flags=re.IGNORECASE)
    # Remove standalone 1947 (UIDAI helpline)
    clean = re.sub(r'(?<!\d)1947(?!\d)', '', clean)

    # ── Step 1: Find UNMASKED 12-digit Aadhaar (spaced as 4-4-4) ──
    spaced = re.findall(r'\b(\d{4})\s+(\d{4})\s+(\d{4})\b', clean)
    if spaced:
        number = ''.join(spaced[0])
        print(f"  ✅ Aadhaar regex (unmasked, spaced): {number}")
        return number, "unmasked"

    # ── Step 2: Find UNMASKED 12-digit Aadhaar (compact, no spaces) ──
    compact = re.findall(r'(?<!\d)(\d{12})(?!\d)', clean)
    if compact:
        print(f"  ✅ Aadhaar regex (unmasked, compact): {compact[0]}")
        return compact[0], "unmasked"

    # ── Step 3: Find MASKED Aadhaar with X prefix pattern ──
    # Matches: "XXXX XXXX 6941" or "xxxx xxxx 6941" or "XXXXXXXX6941"
    x_masked = re.findall(
        r'[Xx*]{4}\s*[Xx*]{4}\s*(\d{4})\b',
        clean,
        flags=re.IGNORECASE
    )
    if x_masked:
        print(f"  ✅ Aadhaar regex (masked, X-prefix): {x_masked[0]}")
        return x_masked[0], "masked"

    # ── Step 4: Find MASKED standalone 4-digit on its own line ──
    # Used by Tinu Verma ("6941") and Mohit Gupta ("8627")
    # Must be EXACTLY 4 digits on the line, not part of a date (19xx/20xx)
    for line in clean.split('\n'):
        stripped = line.strip()
        if re.fullmatch(r'\d{4}', stripped):
            # Exclude year-like values
            if not re.match(r'(19|20)\d{2}', stripped):
                print(f"  ✅ Aadhaar regex (masked, standalone 4-digit): {stripped}")
                return stripped, "masked"

    print("  ❌ Aadhaar number not found via regex")
    return "N/A", "masked"


# =========================
# REGEX: PAN NUMBER
# =========================

def extract_pan_regex(text):
    """
    Extract PAN number using regex.
    PAN format: 5 letters + 4 digits + 1 letter  e.g. ABCDE1234F
    """
    if not text:
        return "N/A"

    matches = re.findall(r'\b[A-Z]{5}[0-9]{4}[A-Z]\b', text.upper())
    if matches:
        print(f"  ✅ PAN via regex: {matches[0]}")
        return matches[0]

    print("  ❌ No PAN found via regex")
    return "N/A"


# =========================
# LLM: AADHAAR DETAILS (name, dob, address ONLY — no number)
# =========================

def extract_aadhaar_details_llm(text):
    """
    LLM extracts name, dob, address from Aadhaar card text.
    Aadhaar NUMBER is intentionally excluded — handled by regex.
    """
    if not text:
        return {}

    prompt = f"""
You are extracting details from an Aadhaar card OCR text.
Return ONLY a valid JSON object with exactly these keys:

{{
  "name": "card holder full name in English",
  "dob": "date of birth DD/MM/YYYY",
  "address": "complete address in English"
}}

RULES:
- "name": Card holder's personal name only, in English.
  It appears right after "Government of India" / "भारत सरकार" line.
  Do NOT include issuer names, UIDAI, or Government of India.
- "dob": Look for "DOB:", "Date of Birth", "जन्म तिथि". Format: DD/MM/YYYY.
- "address": English address only. Look for "Address:" section.
  Include S/O, D/O, C/O relationship prefix if present.
- Use "N/A" for any field not found.
- Return ONLY raw JSON. No markdown, no explanation.

OCR Text:
{text}
"""

    try:
        res = ollama.chat(
            model="mistral:latest",
            messages=[{"role": "user", "content": prompt}]
        )
        out = res["message"]["content"].strip()
        match = re.search(r"\{.*\}", out, re.DOTALL)
        if not match:
            print("  ❌ No JSON in LLM output (Aadhaar details)")
            return {}
        return json.loads(match.group())
    except Exception as e:
        print(f"  LLM ERROR (Aadhaar details): {e}")
        return {}


# =========================
# LLM: PAN CARD HOLDER NAME ONLY
# =========================

def extract_pan_name_llm(text):
    """
    LLM extracts ONLY the primary card holder's name from PAN card.
    Explicitly told NOT to return father's name.
    PAN number handled by regex separately.
    """
    if not text:
        return {}

    prompt = f"""
You are extracting the card holder's name from a PAN card OCR text.
Return ONLY a valid JSON object:

{{
  "name": "card holder name"
}}

RULES:
- Return ONLY the card holder's own name (नाम / Name field).
- The name appears after the label "नाम / Name" on the card.
- Do NOT return the father's name — that appears after "पिता का नाम / Father's Name".
- Do NOT return "Income Tax Department", "Government of India", or issuer text.
- Return only ONE single name, not multiple names separated by commas.
- Use "N/A" if not found.
- Return ONLY raw JSON. No markdown, no explanation.

OCR Text:
{text}
"""

    try:
        res = ollama.chat(
            model="mistral:latest",
            messages=[{"role": "user", "content": prompt}]
        )
        out = res["message"]["content"].strip()
        match = re.search(r"\{.*\}", out, re.DOTALL)
        if not match:
            print("  ❌ No JSON in LLM output (PAN name)")
            return {}
        return json.loads(match.group())
    except Exception as e:
        print(f"  LLM ERROR (PAN name): {e}")
        return {}


# =========================
# PROCESS SINGLE ROW
# =========================

def process_row(row):
    print(f"\n{'='*60}")
    print(f"Processing: {row['Name']}")
    print(f"{'='*60}")

    aadhaar_file = get_txt_filename(row["DMS ID (aadhaar)"])
    pan_file     = get_txt_filename(row["DMS ID (pan)"])

    aadhaar_name = aadhaar_dob = aadhaar_address = "N/A"
    aadhaar_number_extracted = "N/A"
    aadhaar_masked_status    = "masked"
    pan_number = pan_name    = "N/A"

    # ── AADHAAR ──────────────────────────────────────────────────
    if aadhaar_file:
        path = DMS_FOLDER / aadhaar_file
        print(f"  [Aadhaar] {path}")

        if path.exists():
            text = read_text_file(path)
            print(f"  [Aadhaar] {len(text)} chars")

            # NUMBER: regex (reliable)
            aadhaar_number_extracted, aadhaar_masked_status = extract_aadhaar_regex(text)

            # NAME / DOB / ADDRESS: LLM
            data = extract_aadhaar_details_llm(text)
            aadhaar_name    = data.get("name",    "N/A") or "N/A"
            aadhaar_dob     = data.get("dob",     "N/A") or "N/A"
            aadhaar_address = data.get("address", "N/A") or "N/A"

        else:
            print(f"  ❌ File not found: {path}")

    # ── PAN ──────────────────────────────────────────────────────
    if pan_file:
        path = DMS_FOLDER / pan_file
        print(f"  [PAN] {path}")

        if path.exists():
            text = read_text_file(path)
            print(f"  [PAN] {len(text)} chars")

            # NUMBER: regex
            pan_number = extract_pan_regex(text)

            # NAME: LLM (card holder only, not father's name)
            pan_data = extract_pan_name_llm(text)
            pan_name = pan_data.get("name", "N/A") or "N/A"
            print(f"  ✅ PAN Name: {pan_name}")

        else:
            print(f"  ❌ File not found: {path}")

    # ── NORMALIZE ────────────────────────────────────────────────
    name_input = normalize_name(row["Name"])
    name_ocr   = normalize_name(aadhaar_name)

    dob_input  = normalize_dob(row["DOB"])
    dob_ocr    = normalize_dob(str(aadhaar_dob).replace(" 00:00:00", "").strip())

    addr_input = normalize_address(row["Address"])
    addr_ocr   = normalize_address(aadhaar_address)

    pan_input      = str(row["PAN Number"]).strip().upper()
    pan_ocr        = str(pan_number).strip().upper()
    pan_name_input = normalize_name(row["Name"])
    pan_name_ocr   = normalize_name(pan_name)

    # ── RETURN ───────────────────────────────────────────────────
    return {
        "OCR Aadhar Name":          aadhaar_name,
        "Match % (Aadhar Name)":    match_pct(name_input, name_ocr),

        "OCR Aadhar DOB":           aadhaar_dob,
        "Match % (Aadhar DOB)":     match_pct(dob_input, dob_ocr),

        "OCR Aadhar Address":       addr_ocr,
        "Match % (Aadhar Address)": match_pct(addr_input, addr_ocr),

        "masked/unmasked":          aadhaar_masked_status,
        "aadhar number":            aadhaar_number_extracted,

        "PAN Number OCR":           pan_number,
        "Match % (PAN Number)":     100 if pan_input == pan_ocr and pan_input != "N/A" else 0,

        "PAN Name as per OCR":      pan_name,
        "Match % (PAN Name)":       match_pct(pan_name_input, pan_name_ocr),
    }


# =========================
# MAIN
# =========================

def main():
    print(f"📂 Input: {INPUT_FILE}")
    df = pd.read_excel(INPUT_FILE)
    print(f"✅ Loaded {len(df)} rows\n")

    results = []
    for idx, row in df.iterrows():
        try:
            result = process_row(row)
        except Exception as e:
            print(f"  ❌ ERROR row {idx} ({row.get('Name','?')}): {e}")
            result = {
                "OCR Aadhar Name": "ERROR",    "Match % (Aadhar Name)": 0,
                "OCR Aadhar DOB": "ERROR",     "Match % (Aadhar DOB)": 0,
                "OCR Aadhar Address": "ERROR", "Match % (Aadhar Address)": 0,
                "masked/unmasked": "masked",   "aadhar number": "N/A",
                "PAN Number OCR": "ERROR",     "Match % (PAN Number)": 0,
                "PAN Name as per OCR": "ERROR","Match % (PAN Name)": 0,
            }
        results.append(result)

    final_df = pd.concat([df, pd.DataFrame(results)], axis=1)
    final_df.to_excel(OUTPUT_FILE, index=False)
    print(f"\n✅ DONE → {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

In [ ]:
# FILE MERGING + FINAL REPORT

current_timestamp = datetime.now().strftime("%Y-%m-%d %H-%M-%S")


input_file_path1 = r"input_file.xlsx"
input_file_path2 = OUTPUT_FILE_path

final_output_path = fr"final_OCR_report_{current_timestamp}.xlsx"

# =========================
# LOAD FILES
# =========================
df_input = pd.read_excel(input_file_path1)
df_output = pd.read_excel(input_file_path2)

# =========================
# CLEAN COLUMN NAMES (fix hidden bugs)
# =========================
df_input.columns = df_input.columns.str.strip()
df_output.columns = df_output.columns.str.strip()

# =========================
# SHEET 1 → INPUT AS IS
# =========================
sheet1_df = df_input.copy()

# =========================
# SHEET 2 → PROCESSED DATA (FROM OUTPUT FILE)
# =========================
# sheet2_cols = [
#     'S. No.', 'CIF', 'Name', 'DOB', 'Address', 'PAN Number',
#     'DMS ID (aadhaar)', 'Document Type', 'DMS ID (pan)',
#     'Document Type',  # duplicate handled automatically
#     'OCR Aadhar Name', 'OCR Aadhar DOB', 'OCR Aadhar Address',
#     'masked/unmasked', 'PAN Number OCR', 'PAN Name as per OCR'
# ]
sheet2_cols = [
    'S. No.', 'CIF',
    'DMS ID (aadhaar)', 'Document Type', 'DMS ID (pan)',
    'Document Type',  # duplicate handled automatically
    'OCR Aadhar Name', 'OCR Aadhar DOB', 'OCR Aadhar Address',
    'masked/unmasked', 'PAN Number OCR', 'PAN Name as per OCR'
]

# remove duplicates safely
sheet2_cols = list(dict.fromkeys(sheet2_cols))

# keep only available columns
sheet2_cols = [col for col in sheet2_cols if col in df_output.columns]

sheet2_df = df_output[sheet2_cols].copy()

# =========================
# SHEET 3 → MATCH %
# =========================
sheet3_cols = [
    'S. No.', 'CIF',
    'Match % (Aadhar Name)', 'Match % (Aadhar DOB)',
    'Match % (Aadhar Address)',
    'Match % (PAN Number)', 'Match % (PAN Name)'
]

sheet3_cols = [col for col in sheet3_cols if col in df_output.columns]

sheet3_df = df_output[sheet3_cols].copy()

# =========================
# WRITE EXCEL (3 SHEETS)
# =========================
with pd.ExcelWriter(final_output_path, engine='openpyxl') as writer:
    sheet1_df.to_excel(writer, sheet_name='Input', index=False)
    sheet2_df.to_excel(writer, sheet_name='OCR Results', index=False)
    sheet3_df.to_excel(writer, sheet_name='Reports', index=False)

print("Done:", final_output_path)
if os.path.exists(final_output_path):
    os.startfile(final_output_path)
else:
    print(f"File not found: {final_output_path}")